In [10]:
import os
import sys
import subprocess
import ctypes
import sys as sys_lib

PROJECT_ROOT = '/Users/doanvinhnhan/Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import thư viện cốt lõi của Manim
from manim import *

# 3. Import toàn bộ thư viện custom của bạn

from skills.fami_lib import *
from skills.fami_assets_helper import *
from skills.fami_effects import * # Import thêm file này phòng trường hợp các hiệu ứng nằm ở đây
from skills.bit import BitSequence
from skills.broadcasting import Broadcasting
from skills.receiving import Receiving
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")

TexTemplate(_body='', tex_compiler='latex', description='', output_format='.dvi', documentclass='\\documentclass[preview]{standalone}', preamble='\\usepackage[english]{babel}\n\\usepackage{amsmath}\n\\usepackage{amssymb}\n\\usepackage[utf8]{vietnam}', placeholder_text='YourTextHere', post_doc_commands='')

In [17]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache --media_dir /Users/doanvinhnhan/Roo-Code/media Scene2MoHinhHoa

from scipy import stats
import math
import numpy as np

class Scene2MoHinhHoa(FaMIBaseScene):
    def construct(self):
        self.play(Write(Text("START TEST")))
        self.wait(1)

        # ==========================================
        # CHUẨN BỊ HỆ TRỤC + ĐIỂM B
        # ==========================================

        grid = NumberPlane(
            x_range=[-8, 8, 1],
            y_range=[-4.5, 4.5, 1],
            background_line_style={
                "stroke_color": TEAL,
                "stroke_width": 2,
                "stroke_opacity": 0.3
            },
            axis_config={
                "stroke_width":0,
                "include_tip":False,
            }
        )

        fa_title = VGroup(
            Text("TƯƠNG QUAN SAI SỐ", font="Arial", weight="BOLD", font_size=36),
            Text("VỊ TRÍ ROBOT", font="Arial", font_size=24)
        ).arrange(DOWN)
        fa_title.move_to(UP * 5.5) 

        axes = Axes(
            x_range=[-0.5, 4.5, 1],
            y_range=[-0.5, 4.5, 1],
            x_length=6.5,
            y_length=6.5,
            axis_config={
                "include_tip": True,
                "include_numbers": True,
                "font_size": 24,
                "tip_length": 0.2,
            },
        ).move_to(ORIGIN + LEFT * 0.5)

        x_label = axes.get_x_axis_label(MathTex("X"), direction=RIGHT)
        y_label = axes.get_y_axis_label(MathTex("Y"), direction=UP)

        pt_B    = axes.c2p(2, 2)
        dot_B   = Dot(pt_B, color=GREEN, radius=0.13)
        
        label_B = Text(
            "Điểm đến mục tiêu là (2,2)", 
            font_size=20, 
            font="Arial", 
            t2c={"(2,2)": GREEN} 
        )
        label_B.move_to(axes.c2p(3, 4))

        self.play(
            FadeIn(axes),
            FadeIn(x_label),
            FadeIn(y_label),
            FadeIn(dot_B),
            Write(label_B)
        )
        
        # ==========================================
        # CHUẨN BỊ: 45 chấm scatter quanh B
        # ==========================================

        np.random.seed(42)
        scatter_offsets = np.random.normal(0, 0.35, size=(45, 2))
        scatter_dots = VGroup(*[Dot(axes.c2p(2 + dx, 2 + dy), color=RED, radius=0.13, fill_opacity=0.3) for dx, dy in scatter_offsets])

        # ==========================================
        # CHUẨN BỊ CÁC CÔNG THỨC (CÁCH DÙNG MATRIX CỦA MANIM)
        # ==========================================

        # --- 1. Công thức P ---
        title_p = Text("Công thức vị trí của robot:", font_size=24, font="Arial", weight="BOLD")
        
        vec_p = MathTex(r"\vec{p} = ", font_size=36)
        mat_p = Matrix([["X"], ["Y"]]).scale(0.7)
        mat_p.get_entries()[0].set_color(YELLOW)
        mat_p.get_entries()[1].set_color(RED)
        formula_p = VGroup(vec_p, mat_p).arrange(RIGHT, buff=0.15)

        desc_X = VGroup(MathTex("X", color=YELLOW), Text(": tọa độ trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_Y = VGroup(MathTex("Y", color=RED), Text(": tọa độ trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_p = VGroup(desc_X, desc_Y).arrange(DOWN, aligned_edge=LEFT)
        content_p = VGroup(formula_p, desc_group_p).arrange(RIGHT, buff=1.0)
        group_p = VGroup(title_p, content_p).arrange(DOWN, aligned_edge=LEFT)

        # --- 2. Công thức Mu ---
        title_mu = Text("Công thức vị trí mục tiêu kỳ vọng:", font_size=24, font="Arial", weight="BOLD")
        
        vec_mu = MathTex(r"\vec{\mu} = ", font_size=36)
        mat_mu = Matrix([[r"\mu_X"], [r"\mu_Y"]]).scale(0.7)
        mat_mu.get_entries()[0].set_color(GREEN)
        mat_mu.get_entries()[1].set_color(GREEN)
        formula_mu = VGroup(vec_mu, mat_mu).arrange(RIGHT, buff=0.15)

        desc_mu_X = VGroup(MathTex(r"\mu_X", color=GREEN), Text(": mục tiêu trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_mu_Y = VGroup(MathTex(r"\mu_Y", color=GREEN), Text(": mục tiêu trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_mu = VGroup(desc_mu_X, desc_mu_Y).arrange(DOWN, aligned_edge=LEFT)
        content_mu = VGroup(formula_mu, desc_group_mu).arrange(RIGHT, buff=1.0)
        group_mu = VGroup(title_mu, content_mu).arrange(DOWN, aligned_edge=LEFT)

        # --- 3. Công thức Epsilon ---
        title_eps = Text("Công thức vectơ sai số:", font_size=24, font="Arial", weight="BOLD")
        
        vec_eps = MathTex(r"\vec{\varepsilon} = \vec{p} - \vec{\mu} = ", font_size=36)
        mat_eps = Matrix([[r"\varepsilon_X"], [r"\varepsilon_Y"]]).scale(0.7)
        mat_eps.get_entries()[0].set_color(YELLOW)
        mat_eps.get_entries()[1].set_color(RED)
        formula_eps = VGroup(vec_eps, mat_eps).arrange(RIGHT, buff=0.15)

        desc_eps_X = VGroup(MathTex(r"\varepsilon_X", color=YELLOW), Text(": sai số trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_eps_Y = VGroup(MathTex(r"\varepsilon_Y", color=RED), Text(": sai số trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_eps = VGroup(desc_eps_X, desc_eps_Y).arrange(DOWN, aligned_edge=LEFT)
        content_eps = VGroup(formula_eps, desc_group_eps).arrange(RIGHT, buff=1.0)
        group_eps = VGroup(title_eps, content_eps).arrange(DOWN, aligned_edge=LEFT)

        # --- 4. Công thức Phân phối ---
        title_dist = Text("Phân phối của vectơ sai số:", font_size=24, font="Arial", weight="BOLD")
        # Đã lược bỏ bớt khoảng trắng đặc biệt \! để đảm bảo an toàn tuyệt đối
        formula_dist = MathTex(
            r"\vec{\varepsilon} \sim \mathcal{N}(\vec{0}, \Sigma)",
            font_size=36
        )
        
        group_dist = VGroup(title_dist, formula_dist).arrange(DOWN, aligned_edge=LEFT)

        # Gom tất cả vào một cụm và căn giữa màn hình
        all_formulas = VGroup(group_p, group_mu, group_eps, group_dist).arrange(DOWN, buff=0.5, aligned_edge=LEFT)
        all_formulas.shift(RIGHT * 2)
        self.add(all_formulas)
        
        # ==========================================
        # PHÂN ĐOẠN 1: MÔ PHỎNG SCATTER ĐIỂM DỪNG
        # ==========================================

        text1a = "Để trả lời, chúng ta cần nhìn bài toán bằng ngôn ngữ xác suất thống kê."
        with self.voiceover(text=text1a) as t:
            self.update_subtitle(text1a)
            self.play(
                FadeIn(grid),
                FadeIn(fa_title),
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[:15]], lag_ratio=0.1),
                run_time=max(t.duration, 1.5)
            )

        text1b = "Điểm thực tế mà robot dừng lại không phải một điểm cố định mà là một biến ngẫu nhiên."
        with self.voiceover(text=text1b) as t:
            self.update_subtitle(text1b)
            self.play(
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[15:30]], lag_ratio=0.1),
                run_time=max(t.duration, 1.5)
            )

        text1c = "Nguyên nhân là do nhiễu động cơ, ma sát sàn, rung lắc cơ khí."
        with self.voiceover(text=text1c) as t:
            self.update_subtitle(text1c)
            self.play(
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[30:]], lag_ratio=0.1),
                run_time=max(t.duration - 0.5, 1.5)
            )
            self.play(
                *[Flash(d.get_center(), color=RED, flash_radius=0.15, num_lines=4, line_length=0.1) 
                    for d in scatter_dots[::4]], 
                run_time=0.5
            )

        self.wait(0.5)

        # ==========================================
        # CHUYỂN CẢNH: XÓA ĐỒ THỊ
        # ==========================================
        
        self.play(
            FadeOut(axes), FadeOut(x_label), FadeOut(y_label),
            FadeOut(dot_B), FadeOut(label_B), FadeOut(scatter_dots),
            run_time=1.0
        )
        self.wait(0.5)

        # ==========================================
        # PHÂN ĐOẠN 2: GIẢI THÍCH CÔNG THỨC 
        # ==========================================

        # --- Giải thích vị trí P ---
        with self.voiceover(text="Ta biểu diễn vị trí đó") as t:
            self.update_subtitle("Ta biểu diễn vị trí đó")
            self.play(Write(title_p), Write(formula_p), run_time=max(t.duration, 0.8))

        with self.voiceover(text="bằng một vectơ ngẫu nhiên hai chiều:") as t:
            self.update_subtitle("bằng một vectơ ngẫu nhiên hai chiều:")
            self.wait(t.duration)

        with self.voiceover(text="X là toạ độ trục hoành,") as t:
            self.update_subtitle("X là toạ độ trục hoành,")
            self.play(Write(desc_X), run_time=max(t.duration, 0.6))

        with self.voiceover(text="Y là toạ độ trục tung.") as t:
            self.update_subtitle("Y là toạ độ trục tung.")
            self.play(Write(desc_Y), run_time=max(t.duration, 0.6))

        # --- Giải thích mục tiêu Mu ---
        with self.voiceover(text="Điểm mục tiêu lý thuyết —") as t:
            self.update_subtitle("Điểm mục tiêu lý thuyết —")
            self.play(Write(title_mu), Write(formula_mu), run_time=max(t.duration, 0.8))

        with self.voiceover(text="nơi robot lẽ ra phải đến —") as t:
            self.update_subtitle("nơi robot lẽ ra phải đến —")
            self.wait(t.duration)

        with self.voiceover(text="chính là vectơ kỳ vọng mu.") as t:
            self.update_subtitle("chính là vectơ kỳ vọng mu.")
            self.play(Write(desc_mu_X), Write(desc_mu_Y), run_time=max(t.duration, 0.7))

        # --- Giải thích sai số Epsilon ---
        with self.voiceover(text="Và sai số vị trí epsilon") as t:
            self.update_subtitle("Và sai số vị trí epsilon")
            self.play(Write(title_eps), Write(formula_eps), run_time=max(t.duration, 0.8))

        with self.voiceover(text="là hiệu giữa vị trí thực tế và kỳ vọng,") as t:
            self.update_subtitle("là hiệu giữa vị trí thực tế và kỳ vọng,")
            self.wait(t.duration)

        with self.voiceover(text="gồm thành phần epsilon X") as t:
            self.update_subtitle("gồm thành phần epsilon X")
            self.play(Write(desc_eps_X), run_time=max(t.duration, 0.6))

        with self.voiceover(text="và thành phần epsilon Y.") as t:
            self.update_subtitle("và thành phần epsilon Y.")
            self.play(Write(desc_eps_Y), run_time=max(t.duration, 0.6))

        # --- Giải thích Phân phối chuẩn ---
        with self.voiceover(text="Ta giả thiết vectơ sai số") as t:
            self.update_subtitle("Ta giả thiết vectơ sai số")
            self.play(Write(title_dist), Write(formula_dist), run_time=max(t.duration, 0.8))

        with self.voiceover(text="tuân theo phân phối chuẩn hai chiều") as t:
            self.update_subtitle("tuân theo phân phối chuẩn hai chiều")
            self.wait(t.duration)

        with self.voiceover(text="với ma trận hiệp phương sai Sigma.") as t:
            self.update_subtitle("với ma trận hiệp phương sai Sigma.")
            self.wait(t.duration)

        self.wait(1.5)

        # ==========================================
        # FADE OUT TOÀN BỘ — kết thúc Scene 2
        # ==========================================
        
        self.play(FadeOut(all_formulas), run_time=0.3)
        self.wait(1.5)

Manim Community v0.19.0

KeyboardInterrupt: 

In [20]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 15 --flush_cache --media_dir /Users/doanvinhnhan/Roo-Code/media Scene2MoHinhHoa

from scipy import stats
import math
import numpy as np

class Scene2MoHinhHoa(FaMIBaseScene):
    def construct(self):
        self.play(Write(Text("START TEST")))
        self.wait(1)

        # ==========================================
        # CHUẨN BỊ HỆ TRỤC + ĐIỂM B
        # ==========================================

        grid = NumberPlane(
            x_range=[-8, 8, 1],
            y_range=[-4.5, 4.5, 1],
            background_line_style={
                "stroke_color": TEAL,
                "stroke_width": 2,
                "stroke_opacity": 0.3
            },
            axis_config={
                "stroke_width":0,
                "include_tip":False,
            }
        )

        # Đã thay bằng Text thuần túy để tránh lỗi LaTeX nếu hàm create_title dùng Tex()
        fa_title = VGroup(
            Text("TƯƠNG QUAN SAI SỐ", font="Arial", weight="BOLD", font_size=36),
            Text("VỊ TRÍ ROBOT", font="Arial", font_size=24)
        ).arrange(DOWN)
        fa_title.move_to(UP * 5.5) 

        axes = Axes(
            x_range=[-0.5, 4.5, 1],
            y_range=[-0.5, 4.5, 1],
            x_length=6.5,
            y_length=6.5,
            axis_config={
                "include_tip": True,
                "include_numbers": True,
                "font_size": 24,
                "tip_length": 0.2,
            },
        ).move_to(ORIGIN + LEFT * 0.5)

        x_label = axes.get_x_axis_label(MathTex("X"), direction=RIGHT)
        y_label = axes.get_y_axis_label(MathTex("Y"), direction=UP)

        pt_B    = axes.c2p(2, 2)
        dot_B   = Dot(pt_B, color=GREEN, radius=0.13)
        
        label_B = Text(
            "Điểm đến mục tiêu là (2,2)", 
            font_size=20, 
            font="Arial", 
            t2c={"(2,2)": GREEN} 
        )
        label_B.move_to(axes.c2p(3, 4))

        self.play(
            FadeIn(axes),
            FadeIn(x_label),
            FadeIn(y_label),
            FadeIn(dot_B),
            Write(label_B)
        )
        
        # ==========================================
        # CHUẨN BỊ: 45 chấm scatter quanh B
        # ==========================================

        np.random.seed(42)
        scatter_offsets = np.random.normal(0, 0.35, size=(45, 2))
        scatter_dots = VGroup(*[Dot(axes.c2p(2 + dx, 2 + dy), color=RED, radius=0.13, fill_opacity=0.3) for dx, dy in scatter_offsets])

        # ==========================================
        # CHUẨN BỊ CÁC CÔNG THỨC (BỐ CỤC DỌC TỪ TRÊN XUỐNG)
        # ==========================================

        # --- 1. Công thức P ---
        title_p = Text("Công thức vị trí của robot:", font_size=24, font="Arial", weight="BOLD")
        
        vec_p = MathTex(r"\vec{p} = ", font_size=36)
        mat_p = Matrix([["X"], ["Y"]]).scale(0.7)
        mat_p.get_entries()[0].set_color(YELLOW)
        mat_p.get_entries()[1].set_color(RED)
        formula_p = VGroup(vec_p, mat_p).arrange(RIGHT, buff=0.15)

        desc_X = VGroup(MathTex("X", color=YELLOW), Text(": tọa độ trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_Y = VGroup(MathTex("Y", color=RED), Text(": tọa độ trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_p = VGroup(desc_X, desc_Y).arrange(DOWN, aligned_edge=LEFT)
        content_p = VGroup(formula_p, desc_group_p).arrange(RIGHT, buff=1.0)
        group_p = VGroup(title_p, content_p).arrange(DOWN, aligned_edge=LEFT)

        # --- 2. Công thức Mu ---
        title_mu = Text("Công thức vị trí mục tiêu kỳ vọng:", font_size=24, font="Arial", weight="BOLD")
        
        vec_mu = MathTex(r"\vec{\mu} = ", font_size=36)
        mat_mu = Matrix([[r"\mu_X"], [r"\mu_Y"]]).scale(0.7)
        mat_mu.get_entries()[0].set_color(GREEN)
        mat_mu.get_entries()[1].set_color(GREEN)
        formula_mu = VGroup(vec_mu, mat_mu).arrange(RIGHT, buff=0.15)

        desc_mu_X = VGroup(MathTex(r"\mu_X", color=GREEN), Text(": mục tiêu trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_mu_Y = VGroup(MathTex(r"\mu_Y", color=GREEN), Text(": mục tiêu trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_mu = VGroup(desc_mu_X, desc_mu_Y).arrange(DOWN, aligned_edge=LEFT)
        content_mu = VGroup(formula_mu, desc_group_mu).arrange(RIGHT, buff=1.0)
        group_mu = VGroup(title_mu, content_mu).arrange(DOWN, aligned_edge=LEFT)

        # --- 3. Công thức Epsilon ---
        title_eps = Text("Công thức vectơ sai số:", font_size=24, font="Arial", weight="BOLD")
        
        vec_eps = MathTex(r"\vec{\varepsilon} = \vec{p} - \vec{\mu} = ", font_size=36)
        mat_eps = Matrix([[r"\varepsilon_X"], [r"\varepsilon_Y"]]).scale(0.7)
        mat_eps.get_entries()[0].set_color(YELLOW)
        mat_eps.get_entries()[1].set_color(RED)
        formula_eps = VGroup(vec_eps, mat_eps).arrange(RIGHT, buff=0.15)

        desc_eps_X = VGroup(MathTex(r"\varepsilon_X", color=YELLOW), Text(": sai số trục hoành", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)
        desc_eps_Y = VGroup(MathTex(r"\varepsilon_Y", color=RED), Text(": sai số trục tung", font_size=20, font="Arial")).arrange(RIGHT, buff=0.15)

        desc_group_eps = VGroup(desc_eps_X, desc_eps_Y).arrange(DOWN, aligned_edge=LEFT)
        content_eps = VGroup(formula_eps, desc_group_eps).arrange(RIGHT, buff=1.0)
        group_eps = VGroup(title_eps, content_eps).arrange(DOWN, aligned_edge=LEFT)

        # --- 4. Công thức Phân phối ---
        title_dist = Text("Phân phối của vectơ sai số:", font_size=24, font="Arial", weight="BOLD")
        formula_dist = MathTex(
            r"\vec{\varepsilon} \sim \mathcal{N}\!\left(\vec{0},\ \Sigma\right)",
            font_size=36
        )
        
        group_dist = VGroup(title_dist, formula_dist).arrange(DOWN, aligned_edge=LEFT)

        # Gom tất cả vào một cụm và căn giữa màn hình
        all_formulas = VGroup(group_p, group_mu, group_eps, group_dist).arrange(DOWN, buff=0.5, aligned_edge=LEFT)
        all_formulas.shift(RIGHT * 2)
        self.add(all_formulas)
        
        # ==========================================
        # PHÂN ĐOẠN 1: MÔ PHỎNG SCATTER ĐIỂM DỪNG
        # ==========================================

        text1a = "Để trả lời, chúng ta cần nhìn bài toán bằng ngôn ngữ xác suất thống kê."
        with self.voiceover(text=text1a) as t:
            self.update_subtitle(text1a)
            self.play(
                FadeIn(grid),
                FadeIn(fa_title),
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[:15]], lag_ratio=0.1),
                run_time=max(t.duration, 1.5)
            )

        text1b = "Điểm thực tế mà robot dừng lại không phải một điểm cố định mà là một biến ngẫu nhiên."
        with self.voiceover(text=text1b) as t:
            self.update_subtitle(text1b)
            self.play(
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[15:30]], lag_ratio=0.1),
                run_time=max(t.duration, 1.5)
            )

        text1c = "Nguyên nhân là do nhiễu động cơ, ma sát sàn, rung lắc cơ khí."
        with self.voiceover(text=text1c) as t:
            self.update_subtitle(text1c)
            self.play(
                LaggedStart(*[FadeIn(d, scale=1.4) for d in scatter_dots[30:]], lag_ratio=0.1),
                run_time=max(t.duration - 0.5, 1.5)
            )
            self.play(
                *[Flash(d.get_center(), color=RED, flash_radius=0.15, num_lines=4, line_length=0.1) 
                    for d in scatter_dots[::4]], 
                run_time=0.5
            )

        self.wait(0.5)

        # ==========================================
        # CHUYỂN CẢNH: XÓA ĐỒ THỊ
        # ==========================================
        
        self.play(
            FadeOut(axes), FadeOut(x_label), FadeOut(y_label),
            FadeOut(dot_B), FadeOut(label_B), FadeOut(scatter_dots),
            run_time=1.0
        )
        self.wait(0.5)

        # ==========================================
        # PHÂN ĐOẠN 2: GIẢI THÍCH CÔNG THỨC (CÁCH MỚI)
        # ==========================================

        # --- Giải thích vị trí P ---
        with self.voiceover(text="Ta biểu diễn vị trí đó") as t:
            self.update_subtitle("Ta biểu diễn vị trí đó")
            self.play(Write(title_p), Write(formula_p), run_time=max(t.duration, 0.8))

        with self.voiceover(text="bằng một vectơ ngẫu nhiên hai chiều:") as t:
            self.update_subtitle("bằng một vectơ ngẫu nhiên hai chiều:")
            self.wait(t.duration)

        with self.voiceover(text="X là toạ độ trục hoành,") as t:
            self.update_subtitle("X là toạ độ trục hoành,")
            self.play(Write(desc_X), run_time=max(t.duration, 0.6))

        with self.voiceover(text="Y là toạ độ trục tung.") as t:
            self.update_subtitle("Y là toạ độ trục tung.")
            self.play(Write(desc_Y), run_time=max(t.duration, 0.6))

        # --- Giải thích mục tiêu Mu ---
        with self.voiceover(text="Điểm mục tiêu lý thuyết —") as t:
            self.update_subtitle("Điểm mục tiêu lý thuyết —")
            self.play(Write(title_mu), Write(formula_mu), run_time=max(t.duration, 0.8))

        with self.voiceover(text="nơi robot lẽ ra phải đến —") as t:
            self.update_subtitle("nơi robot lẽ ra phải đến —")
            self.wait(t.duration)

        with self.voiceover(text="chính là vectơ kỳ vọng mu.") as t:
            self.update_subtitle("chính là vectơ kỳ vọng mu.")
            self.play(Write(desc_mu_X), Write(desc_mu_Y), run_time=max(t.duration, 0.7))

        # --- Giải thích sai số Epsilon ---
        with self.voiceover(text="Và sai số vị trí epsilon") as t:
            self.update_subtitle("Và sai số vị trí epsilon")
            self.play(Write(title_eps), Write(formula_eps), run_time=max(t.duration, 0.8))

        with self.voiceover(text="là hiệu giữa vị trí thực tế và kỳ vọng,") as t:
            self.update_subtitle("là hiệu giữa vị trí thực tế và kỳ vọng,")
            self.wait(t.duration)

        with self.voiceover(text="gồm thành phần epsilon X") as t:
            self.update_subtitle("gồm thành phần epsilon X")
            self.play(Write(desc_eps_X), run_time=max(t.duration, 0.6))

        with self.voiceover(text="và thành phần epsilon Y.") as t:
            self.update_subtitle("và thành phần epsilon Y.")
            self.play(Write(desc_eps_Y), run_time=max(t.duration, 0.6))

        # --- Giải thích Phân phối chuẩn ---
        with self.voiceover(text="Ta giả thiết vectơ sai số") as t:
            self.update_subtitle("Ta giả thiết vectơ sai số")
            self.play(Write(title_dist), Write(formula_dist), run_time=max(t.duration, 0.8))

        with self.voiceover(text="tuân theo phân phối chuẩn hai chiều") as t:
            self.update_subtitle("tuân theo phân phối chuẩn hai chiều")
            self.wait(t.duration)

        with self.voiceover(text="với ma trận hiệp phương sai Sigma.") as t:
            self.update_subtitle("với ma trận hiệp phương sai Sigma.")
            self.wait(t.duration)

        self.wait(1.5)

        # ==========================================
        # FADE OUT TOÀN BỘ — kết thúc Scene 2
        # ==========================================
        
        self.play(FadeOut(all_formulas), run_time=0.3)
        self.wait(1.5)


Manim Community v0.19.0

[04/11/26 13:04:10] WARNING  Some options were not used: {'shortest': '1', 'metadata':     ]8;id=285701;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=941005;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#801\801]8;;\
                             'comment=Rendered with Manim Community v0.19.0'}                                      